In [8]:

import pandas as pd
import numpy as np

# Import required libraries
try:
    from pgmpy.models import DiscreteBayesianNetwork as BayesianModel
except ImportError:
    from pgmpy.models import BayesianNetwork as BayesianModel

from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination

# Step 1: Load dataset
heartDisease = pd.read_csv("heart.csv")

# Clean column names
heartDisease.columns = heartDisease.columns.str.strip().str.lower()

# Select required columns
columns = [
    "age", "sex", "cp", "trestbps", "chol",
    "fbs", "restecg", "thalach", "exang",
    "oldpeak", "slope", "ca", "thal",
    "heartdisease"
]

heartDisease = heartDisease[columns]

# Convert values to numeric
for col in columns:
    heartDisease[col] = pd.to_numeric(heartDisease[col], errors="coerce")

# Remove missing values
heartDisease.replace("?", np.nan, inplace=True)
heartDisease.dropna(inplace=True)

print("Sample Dataset:")
print(heartDisease.head())

# Step 2: Discretize continuous attributes
heartDisease["age"] = pd.cut(
    heartDisease["age"],
    bins=[0, 40, 55, 100],
    labels=["Young", "Middle", "Old"],
    include_lowest=True
)

heartDisease["chol"] = pd.cut(
    heartDisease["chol"],
    bins=[0, 200, 240, 1000],
    labels=["Normal", "Borderline", "High"],
    include_lowest=True
)

# Convert all columns to strings
for col in columns:
    heartDisease[col] = heartDisease[col].astype(str)

# Step 3: Create Bayesian Network
model = BayesianModel([
    ("age", "heartdisease"),
    ("sex", "heartdisease"),
    ("exang", "heartdisease"),
    ("cp", "heartdisease"),
    ("heartdisease", "restecg"),
    ("heartdisease", "chol")
])

print("\nBayesian Network Structure:")
print(model.edges())

# Step 4: Learn CPDs using Maximum Likelihood Estimator
model.fit(heartDisease)

print("\nModel Check:")
print(model.check_model())

# Display CPDs
print("\nConditional Probability Distributions:")
for cpd in model.get_cpds():
    print(cpd)

# Step 5: Perform inference
heartDisease_infer = VariableElimination(model)

# Query 1
print("\nProbability of Heart Disease given restecg = 1")

q1 = heartDisease_infer.query(
    variables=["heartdisease"],
    evidence={"restecg": "1"},
    show_progress=False
)

print(q1)

# Query 2
print("\nProbability of Heart Disease given cp = 2")

q2 = heartDisease_infer.query(
    variables=["heartdisease"],
    evidence={"cp": "2"},
    show_progress=False
)

print(q2)

print("\nBayesian Network implementation completed successfully!")


Sample Dataset:
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   1       145   233    1        2      150      0      2.3      3   
1   67    1   4       160   286    0        2      108      1      1.5      2   
2   67    1   4       120   229    0        2      129      1      2.6      2   
3   37    1   3       130   250    0        0      187      0      3.5      3   
4   41    0   2       130   204    0        2      172      0      1.4      1   

    ca  thal  heartdisease  
0  0.0   6.0             0  
1  3.0   3.0             2  
2  2.0   7.0             1  
3  0.0   3.0             0  
4  0.0   3.0             0  

Bayesian Network Structure:
[('age', 'heartdisease'), ('heartdisease', 'restecg'), ('heartdisease', 'chol'), ('sex', 'heartdisease'), ('exang', 'heartdisease'), ('cp', 'heartdisease')]

Model Check:
True

Conditional Probability Distributions:
+-------------+-----------+
| age(Middle) | 0.434343  |
+-------------+-------

C:\Users\varun\AppData\Local\Temp\ipykernel_19808\3723168249.py:35: Pandas4Warning: The inplace keyword in DataFrame.dropna is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  heartDisease.dropna(inplace=True)
